In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("../data/raw/nhanes")

file_reports = []
unique_file_paths = []

for file_path in sorted(data_dir.glob("*.xpt")):
    try:
        data = pd.read_sas(
            file_path,
            format="xport",
            encoding="latin-1"
        )

        has_seqn = "SEQN" in data.columns

        if has_seqn:
            row_count = len(data)
            unique_seqn_count = data["SEQN"].nunique()
            duplicated_seqn_count = data["SEQN"].duplicated().sum()
            is_unique = data["SEQN"].is_unique

            if is_unique:
                unique_file_paths.append(file_path)
        else:
            row_count = len(data)
            unique_seqn_count = None
            duplicated_seqn_count = None
            is_unique = False

        file_reports.append({
            "file_name": file_path.name,
            "row_count": row_count,
            "column_count": len(data.columns),
            "has_seqn": has_seqn,
            "unique_seqn_count": unique_seqn_count,
            "duplicated_seqn_count": duplicated_seqn_count,
            "is_unique": is_unique,
            "status": "success"
        })

        print(
            f"{file_path.name}: "
            f"rows={row_count}, "
            f"columns={len(data.columns)}, "
            f"unique_seqn={is_unique}"
        )

    except Exception as error:
        file_reports.append({
            "file_name": file_path.name,
            "status": "error",
            "error_message": str(error)
        })

        print(f"{file_path.name}: ERROR - {error}")

file_report = pd.DataFrame(file_reports)

print()
print("Total XPT files:", len(file_report))
print("Files with unique SEQN:", len(unique_file_paths))

ACQ_L.xpt: rows=11372, columns=5, unique_seqn=True
AGP_L.xpt: rows=2564, columns=3, unique_seqn=True
ALB_CR_L.xpt: rows=8493, columns=8, unique_seqn=True
ALQ_L.xpt: rows=6337, columns=9, unique_seqn=True
AUQ_L.xpt: rows=11744, columns=14, unique_seqn=True
BAQ_L.xpt: rows=6070, columns=15, unique_seqn=True
BAX_L.xpt: rows=4771, columns=45, unique_seqn=True
BIOPRO_L.xpt: rows=7199, columns=42, unique_seqn=True
BMX_L.xpt: rows=8860, columns=22, unique_seqn=True
BPQ_L.xpt: rows=8501, columns=6, unique_seqn=True
BPXO_L.xpt: rows=7801, columns=12, unique_seqn=True
CBC_L.xpt: rows=8727, columns=23, unique_seqn=True
DBQ_L.xpt: rows=11933, columns=27, unique_seqn=True
DEMO_L.xpt: rows=11933, columns=27, unique_seqn=True
DEQ_L.xpt: rows=4305, columns=4, unique_seqn=True
DIQ_L.xpt: rows=11744, columns=9, unique_seqn=True
DPQ_L.xpt: rows=6337, columns=11, unique_seqn=True
DR1IFF_L.xpt: rows=100116, columns=84, unique_seqn=False
DR2IFF_L.xpt: rows=88032, columns=84, unique_seqn=False
DR2TOT_L.xpt: 

In [3]:
import csv

merged_data = None

for file_path in unique_file_paths:
    data = pd.read_sas(
        file_path,
        format="xport",
        encoding="latin-1"
    )

    data = data.copy()

    # Keep SEQN unchanged and prefix all other columns with the file name.
    column_mapping = {
        column: f"{file_path.stem}__{column}"
        for column in data.columns
        if column != "SEQN"
    }

    data = data.rename(columns=column_mapping)

    if merged_data is None:
        merged_data = data
    else:
        merged_data = merged_data.merge(
            data,
            on="SEQN",
            how="outer",
            validate="one_to_one"
        )

    print(
        f"Merged {file_path.name} | "
        f"current shape: {merged_data.shape}"
    )

Merged ACQ_L.xpt | current shape: (11372, 5)
Merged AGP_L.xpt | current shape: (11606, 7)
Merged ALB_CR_L.xpt | current shape: (11606, 14)
Merged ALQ_L.xpt | current shape: (11606, 22)
Merged AUQ_L.xpt | current shape: (11744, 35)
Merged BAQ_L.xpt | current shape: (11744, 49)
Merged BAX_L.xpt | current shape: (11744, 93)
Merged BIOPRO_L.xpt | current shape: (11744, 134)
Merged BMX_L.xpt | current shape: (11877, 155)
Merged BPQ_L.xpt | current shape: (11877, 160)
Merged BPXO_L.xpt | current shape: (11877, 171)
Merged CBC_L.xpt | current shape: (11877, 193)
Merged DBQ_L.xpt | current shape: (11933, 219)
Merged DEMO_L.xpt | current shape: (11933, 245)
Merged DEQ_L.xpt | current shape: (11933, 248)
Merged DIQ_L.xpt | current shape: (11933, 256)
Merged DPQ_L.xpt | current shape: (11933, 266)
Merged DR2TOT_L.xpt | current shape: (11933, 350)
Merged DSQTOT_L.xpt | current shape: (11933, 389)
Merged ECQ_L.xpt | current shape: (11933, 395)
Merged FAR_L.xpt | current shape: (11933, 438)
Merged F

In [4]:
output_dir = Path("../data/processed/nhanes")

output_path = output_dir / "seqn_unique.csv"

merged_data.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    sep=",",
    quoting=csv.QUOTE_MINIMAL,
    quotechar='"',
    doublequote=True,
    lineterminator="\n"
)

print("Output file:", output_path.resolve())
print("Output shape:", merged_data.shape)

Output file: D:\projects\tech-challenge-11iadt-fase-01\data\processed\nhanes\seqn_unique.csv
Output shape: (11933, 862)


In [5]:
saved_data = pd.read_csv(
    output_path,
    encoding="utf-8-sig"
)

print("Saved shape:", saved_data.shape)
print("Duplicated SEQN:", saved_data["SEQN"].duplicated().sum())
print("Output file exists:", output_path.exists())

Saved shape: (11933, 862)
Duplicated SEQN: 0
Output file exists: True


In [6]:
import csv
import pandas as pd
from pathlib import Path

data_dir = Path("../data/processed/nhanes")

input_path = data_dir / "seqn_unique.csv"
output_path = data_dir / "seqn_unique_female.csv"

unique_data = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

gender_column = "DEMO_L__RIAGENDR"

print("Input shape:", unique_data.shape)
print("Gender column exists:", gender_column in unique_data.columns)
print(
    "Gender values:",
    unique_data[gender_column]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

Input shape: (11933, 862)
Gender column exists: True
Gender values: DEMO_L__RIAGENDR
1.0    5575
2.0    6358


In [7]:
female_data = unique_data.loc[
    unique_data[gender_column] == 2.0
    ].copy()

print("Female dataset shape:", female_data.shape)
print("Female participants:", female_data["SEQN"].nunique())
print("Duplicated SEQN:", female_data["SEQN"].duplicated().sum())

Female dataset shape: (6358, 862)
Female participants: 6358
Duplicated SEQN: 0


In [8]:
female_data.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    sep=",",
    quoting=csv.QUOTE_MINIMAL,
    quotechar='"',
    doublequote=True,
    lineterminator="\n"
)

print("Output file:", output_path.resolve())
print("Output file exists:", output_path.exists())

Output file: D:\projects\tech-challenge-11iadt-fase-01\data\processed\nhanes\seqn_unique_female.csv
Output file exists: True


In [9]:
saved_female_data = pd.read_csv(
    output_path,
    encoding="utf-8-sig"
)

print("Saved shape:", saved_female_data.shape)
print(
    "Female participants:",
    saved_female_data["SEQN"].nunique()
)
print(
    "Duplicated SEQN:",
    saved_female_data["SEQN"].duplicated().sum()
)

Saved shape: (6358, 862)
Female participants: 6358
Duplicated SEQN: 0


In [2]:
import pandas as pd

female_dataframe = pd.read_csv("../data/processed/nhanes/seqn_unique_female.csv", encoding="utf-8-sig", low_memory=False)

print(f"Female Dataset: {female_dataframe.shape[0]} linhas e {female_dataframe.shape[1]} colunas")
female_dataframe.head()

Female Dataset: 6358 linhas e 862 colunas


,SEQN,ACQ_L__ACD010A,ACQ_L__ACD010B,ACQ_L__ACD010C,ACQ_L__ACD040,AGP_L__WTPH2YR,AGP_L__LBXAGP,ALB_CR_L__URXUMA,ALB_CR_L__URXUMS,ALB_CR_L__URDUMALC,...,VTQ_L__VTQ261A,VTQ_L__VTD261B,VTQ_L__VTQ271A,VTQ_L__VTD271B,VTQ_L__VTQ281A,VTQ_L__VTD281B,WHQ_L__WHD010,WHQ_L__WHD020,WHQ_L__WHD050,WHQ_L__WHQ070
0,130380.0,NaN,NaN,NaN,2.0,8.532884e+04,1.01,12.43,12.43,5.397605e-79,...,2.0,NaN,1.0,2.0,2.0,NaN,60.0,150.0,165.0,1.0
1,130381.0,1.0,NaN,NaN,NaN,5.397605e-79,NaN,16.12,16.12,5.397605e-79,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,130383.0,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,130385.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,70.0,240.0,240.0,2.0
4,130387.0,1.0,NaN,NaN,NaN,NaN,NaN,5.93,5.93,5.397605e-79,...,NaN,NaN,NaN,NaN,NaN,NaN,67.0,215.0,215.0,2.0


In [3]:
participant_count = female_dataframe["SEQN"].nunique()
column_count = female_dataframe.shape[1]

fully_missing_columns = [
    column
    for column in female_dataframe.columns
    if female_dataframe[column].isna().all()
]

constant_columns = [
    column
    for column in female_dataframe.columns
    if female_dataframe[column].nunique(dropna=True) <= 1
]

missing_rate = female_dataframe.isna().mean()

missing_over_50 = (
    missing_rate[missing_rate > 0.50]
    .sort_values(ascending=False)
)

missing_over_50_report = pd.DataFrame({
    "column": missing_over_50.index,
    "missing_count": [
        female_dataframe[column].isna().sum()
        for column in missing_over_50.index
    ],
    "missing_rate": missing_over_50.values
})

missing_over_50_report["missing_percentage"] = (
        missing_over_50_report["missing_rate"] * 100
).round(2)

text_columns = female_dataframe.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print(f"Participants: {participant_count}")
print(f"Columns: {column_count}")
print(f"Fully empty columns: {len(fully_missing_columns)}")
print(f"Constant columns: {len(constant_columns)}")
print(f"Columns with more than 50% missing: {len(missing_over_50_report)}")
print(f"Text columns: {len(text_columns)}")

print("\nFully empty columns:")
for column in fully_missing_columns:
    print(f"- {column}")

print("\nConstant columns:")
for column in constant_columns:
    print(f"- {column}")

print("\nText columns:")
for column in text_columns:
    print(f"- {column}")

print("\nColumns with more than 50% missing:")
print(
    missing_over_50_report.to_string(index=False)
)

Participants: 6358
Columns: 862
Fully empty columns: 2
Constant columns: 80
Columns with more than 50% missing: 434
Text columns: 8

Fully empty columns:
- BMX_L__BMIHEAD
- IMQ_L__IMQ070

Constant columns:
- ACQ_L__ACD010B
- ACQ_L__ACD010C
- ALB_CR_L__URDUCRLC
- AUQ_L__AUQ410B
- AUQ_L__AUQ410C
- AUQ_L__AUQ410D
- AUQ_L__AUQ410E
- AUQ_L__AUQ410F
- AUQ_L__AUQ410G
- AUQ_L__AUQ410H
- AUQ_L__AUQ410I
- AUQ_L__AUQ410J
- BMX_L__BMIRECUM
- BMX_L__BMIHEAD
- BMX_L__BMILEG
- BMX_L__BMIARML
- BMX_L__BMIARMC
- BMX_L__BMIWAIST
- BMX_L__BMIHIP
- DBQ_L__DBQ073B
- DBQ_L__DBQ073C
- DBQ_L__DBQ073D
- DBQ_L__DBQ073E
- DBQ_L__DBQ073U
- DEMO_L__SDDSRVYR
- DEMO_L__RIAGENDR
- FAR_L__LBDPANLC
- FAR_L__LBDP1ALC
- FAR_L__LBDPRALC
- FAR_L__LBDPDALC
- FAR_L__LBDPHALC
- FAR_L__LBDPD3LC
- FAR_L__LBDPD6LC
- FAR_L__LBDPTALC
- FAR_L__LBDPEDLC
- FAR_L__LBDP1ELC
- FAR_L__LBDPLGLC
- FAR_L__LBDPGHLC
- FAR_L__LBDP1GLC
- FAR_L__LBDPNLLC
- FAR_L__LBDPNRLC
- FAR_L__LBDPOLLC
- FAR_L__LBDPPLLC
- FAR_L__LBDPPMLC
- FAR_L__LBDPSTLC
- 

In [4]:
import csv
from pathlib import Path

import pandas as pd


data_dir = Path("../data/processed/nhanes")

input_path = data_dir / "seqn_unique_female.csv"
output_path = data_dir / "seqn_unique_female_reduced.csv"

female_data = pd.read_csv(
    input_path,
    encoding="utf-8-sig",
    low_memory=False
)

constant_columns = [
    column
    for column in female_data.columns
    if female_data[column].nunique(dropna=True) <= 1
]

missing_rate = female_data.isna().mean()

missing_over_50_columns = (
    missing_rate[missing_rate > 0.50]
    .index
    .tolist()
)

columns_to_remove = sorted(
    set(constant_columns) |
    set(missing_over_50_columns)
)

columns_to_remove = [
    column
    for column in columns_to_remove
    if column != "SEQN"
]

reduced_data = female_data.drop(
    columns=columns_to_remove
)

print("Original shape:", female_data.shape)
print("Constant columns:", len(constant_columns))
print("Columns with more than 50% missing:",len(missing_over_50_columns))
print("Columns to remove:", len(columns_to_remove))
print("Reduced shape:", reduced_data.shape)

print("\nRemoved columns:")
for column in columns_to_remove:
    print(f"- {column}")

Original shape: (6358, 862)
Constant columns: 80
Columns with more than 50% missing: 434
Columns to remove: 464
Reduced shape: (6358, 398)

Removed columns:
- ACQ_L__ACD010B
- ACQ_L__ACD010C
- ACQ_L__ACD040
- AGP_L__LBXAGP
- AGP_L__WTPH2YR
- ALB_CR_L__URDUCRLC
- ALQ_L__ALQ111
- ALQ_L__ALQ121
- ALQ_L__ALQ130
- ALQ_L__ALQ142
- ALQ_L__ALQ151
- ALQ_L__ALQ170
- ALQ_L__ALQ270
- ALQ_L__ALQ280
- AUQ_L__AUQ410A
- AUQ_L__AUQ410B
- AUQ_L__AUQ410C
- AUQ_L__AUQ410D
- AUQ_L__AUQ410E
- AUQ_L__AUQ410F
- AUQ_L__AUQ410G
- AUQ_L__AUQ410H
- AUQ_L__AUQ410I
- AUQ_L__AUQ410J
- BAQ_L__BAQ341
- BAQ_L__BAQ391A
- BAQ_L__BAQ391B
- BAQ_L__BAQ401
- BAQ_L__BAQ421
- BAQ_L__BAQ431
- BAQ_L__BAQ491
- BAQ_L__BAQ550
- BAQ_L__BAQ560
- BAX_L__BAARFC11
- BAX_L__BAARFC12
- BAX_L__BAARFC21
- BAX_L__BAARFC22
- BAX_L__BAARFC31
- BAX_L__BAARFC32
- BAX_L__BAARFC41
- BAX_L__BAARFC42
- BAX_L__BAARFC51
- BAX_L__BAARFC52
- BAX_L__BAQ110
- BAX_L__BAQ121
- BAX_L__BAQ125
- BAX_L__BAQ132
- BAX_L__BAQ140
- BAX_L__BAQ150
- BAX_L__BAQ160
- B

In [5]:
reduced_data.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    sep=",",
    quoting=csv.QUOTE_MINIMAL,
    quotechar='"',
    doublequote=True,
    lineterminator="\n"
)

print("\nOutput file:", output_path.resolve())
print("Output file exists:", output_path.exists())


Output file: D:\projects\tech-challenge-11iadt-fase-01\data\processed\nhanes\seqn_unique_female_reduced.csv
Output file exists: True


In [6]:
saved_reduced_data = pd.read_csv(
    output_path,
    encoding="utf-8-sig",
    low_memory=False
)

print("Saved shape:", saved_reduced_data.shape)
print(
    "Duplicated SEQN:",
    saved_reduced_data["SEQN"].duplicated().sum()
)
print(
    "Remaining columns with more than 50% missing:",
    (saved_reduced_data.isna().mean() > 0.50).sum()
)

Saved shape: (6358, 398)
Duplicated SEQN: 0
Remaining columns with more than 50% missing: 0


In [8]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd


data_dir = Path("../data/processed/nhanes")

input_path = data_dir / "seqn_unique_female_reduced.csv"
output_path = data_dir / "seqn_unique_female_numeric.csv"

female_data = pd.read_csv(
    input_path,
    encoding="utf-8-sig",
    low_memory=False
)

text_columns = female_data.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Text columns before conversion:")
for column in text_columns:
    print(f"- {column}")

time_columns = [
    "SLQ_L__SLQ300",
    "SLQ_L__SLQ310",
    "SLQ_L__SLQ320",
    "SLQ_L__SLQ330"
]

time_features = {}

for column in time_columns:
    parsed_time = pd.to_datetime(
        female_data[column],
        format="%H:%M",
        errors="coerce"
    )

    time_features[f"{column}__minutes"] = (
            parsed_time.dt.hour * 60
            + parsed_time.dt.minute
    )

time_features = pd.DataFrame(
    time_features,
    index=female_data.index
)

female_data = female_data.drop(
    columns=time_columns
)

female_data = pd.concat(
    [female_data, time_features],
    axis=1
)

categorical_columns = [
    column
    for column in text_columns
    if column not in time_columns
]

female_data = pd.get_dummies(
    female_data,
    columns=categorical_columns,
    dummy_na=True,
    dtype="int8"
)

Text columns before conversion:
- BPXO_L__BPAOARM
- LUX_L__LUAPNME
- PAQ_L__PAD790U
- SLQ_L__SLQ300
- SLQ_L__SLQ310
- SLQ_L__SLQ320
- SLQ_L__SLQ330


In [9]:
remaining_text_columns = female_data.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Remaining text columns:", remaining_text_columns)
print("Final shape:", female_data.shape)
print(
    "Duplicated SEQN:",
    female_data["SEQN"].duplicated().sum()
)

Remaining text columns: []
Final shape: (6358, 406)
Duplicated SEQN: 0


In [10]:
print(
    female_data.dtypes
    .value_counts()
    .to_string()
)

float64    395
int8        11


In [11]:
female_data.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    sep=",",
    quoting=csv.QUOTE_MINIMAL,
    quotechar='"',
    doublequote=True,
    lineterminator="\n"
)

print("Output file:", output_path.resolve())
print("Output file exists:", output_path.exists())

Output file: D:\projects\tech-challenge-11iadt-fase-01\data\processed\nhanes\seqn_unique_female_numeric.csv
Output file exists: True


In [13]:
saved_numeric_data = pd.read_csv(
    output_path,
    encoding="utf-8-sig",
    low_memory=False
)

print("Saved shape:", saved_numeric_data.shape)
print(
    "Remaining text columns:",
    saved_numeric_data.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()
)
print(
    "Duplicated SEQN:",
    saved_numeric_data["SEQN"].duplicated().sum()
)

Saved shape: (6358, 406)
Remaining text columns: []
Duplicated SEQN: 0
